In [ ]:
!pip install transformers


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from tqdm import tqdm
import os

class KeywordToSentenceDataset(Dataset):
    def __init__(self, keywords, sentences, tokenizer, max_length=512):
        self.keywords = keywords
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.keywords)

    def __getitem__(self, idx):
        keyword = str(self.keywords[idx])
        sentence = str(self.sentences[idx])

        input_text = f"generate sentence: {keyword}"

        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        target_encoding = self.tokenizer(
            sentence,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': input_encoding['input_ids'].flatten(),
            'attention_mask': input_encoding['attention_mask'].flatten(),
            'labels': target_encoding['input_ids'].flatten()
        }

class T5SentenceGenerator:
    def __init__(self, model_name='t5-small'):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.model.to(self.device)

    def train_from_excel(self, excel_file, epochs=5, batch_size=8, learning_rate=5e-5, save_path="./trained_model"):
        # Read Excel file
        df = pd.read_excel(excel_file)
        sentences = df['text'].tolist()
        keywords = df['extracted_keywords'].tolist()

        # Clean data
        clean_data = [(str(kw).strip(), str(sent).strip())
                     for sent, kw in zip(sentences, keywords)
                     if pd.notna(sent) and pd.notna(kw) and str(sent).strip() and str(kw).strip()]

        keywords, sentences = zip(*clean_data)

        # Split data
        train_kw, val_kw, train_sent, val_sent = train_test_split(
            keywords, sentences, test_size=0.2, random_state=42
        )

        # Create datasets
        train_dataset = KeywordToSentenceDataset(train_kw, train_sent, self.tokenizer)
        val_dataset = KeywordToSentenceDataset(val_kw, val_sent, self.tokenizer)

        # Create data loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Setup optimizer
        optimizer = AdamW(self.model.parameters(), lr=learning_rate)

        # Training loop
        self.model.train()
        best_val_loss = float('inf')

        for epoch in range(epochs):
            total_train_loss = 0

            for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
                optimizer.zero_grad()

                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                total_train_loss += loss.item()

                loss.backward()
                optimizer.step()

            # Validation
            self.model.eval()
            total_val_loss = 0

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['labels'].to(self.device)

                    outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    total_val_loss += outputs.loss.item()

            avg_val_loss = total_val_loss / len(val_loader)

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                os.makedirs(save_path, exist_ok=True)
                self.model.save_pretrained(save_path)
                self.tokenizer.save_pretrained(save_path)

            self.model.train()
            print(f'Epoch {epoch+1}: Val Loss = {avg_val_loss:.4f}')

    def generate_sentence(self, keywords, max_length=100):
        self.model.eval()
        input_text = f"generate sentence: {keywords}"

        inputs = self.tokenizer(input_text, return_tensors='pt', max_length=512, truncation=True).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=max_length, num_beams=4, do_sample=True, temperature=0.9)

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def load_model(self, model_path):
        self.model = T5ForConditionalGeneration.from_pretrained(model_path)
        self.tokenizer = T5Tokenizer.from_pretrained(model_path)
        self.model.to(self.device)

# Usage:
generator = T5SentenceGenerator()
generator.train_from_excel('/content/Text and Keywords.xlsx', epochs=10)
# sentence = generator.generate_sentence('your keywords here')

Epoch 1/10:   0%|          | 1/448 [01:06<8:14:20, 66.36s/it]

In [ ]:
import pandas as pd
df = pd.read_excel('/content/Text and Keywords.xlsx')
print(df.columns)


Index(['text', 'extracted_keywords'], dtype='object')


In [ ]:
!pip install transformers pandas openpyxl


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq

# ====================
# Custom Dataset
# ====================
class TextKeywordDataset(Dataset):
    def __init__(self, dataframe, tokenizer, source_max_len=128, target_max_len=64):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.source_max_len = source_max_len
        self.target_max_len = target_max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        source_text = str(self.data.iloc[idx, 0])  # assuming first col = text
        target_text = str(self.data.iloc[idx, 1])  # assuming second col = keywords

        source = self.tokenizer(
            source_text,
            padding="max_length",
            truncation=True,
            max_length=self.source_max_len,
            return_tensors="pt"
        )
        target = self.tokenizer(
            target_text,
            padding="max_length",
            truncation=True,
            max_length=self.target_max_len,
            return_tensors="pt"
        )

        labels = target["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100  # ignore padding in loss

        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "labels": labels,
        }

# ====================
# Main Trainer Class
# ====================
class T5SentenceGeneratorScratch:
    def __init__(self, model_name="t5-small"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)

        # Define config explicitly (important when training from scratch)
        from transformers import T5Config
        config = T5Config.from_pretrained(model_name)
        config.decoder_start_token_id = self.tokenizer.pad_token_id
        config.eos_token_id = self.tokenizer.eos_token_id
        config.pad_token_id = self.tokenizer.pad_token_id
        config.vocab_size = self.tokenizer.vocab_size

        # Initialize model from scratch
        self.model = T5ForConditionalGeneration(config)

    def train_from_excel(self, excel_path, epochs=5, batch_size=2, learning_rate=5e-4):
        # Load dataset
        df = pd.read_excel(excel_path)

        dataset = TextKeywordDataset(df, self.tokenizer)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        data_collator = DataCollatorForSeq2Seq(self.tokenizer, model=self.model)

        # Training arguments
        training_args = TrainingArguments(
            output_dir="./results",
            overwrite_output_dir=True,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            save_steps=1000,
            save_total_limit=2,
            learning_rate=learning_rate,
            logging_dir="./logs",
            logging_steps=50,
            report_to="none",  # disables wandb
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=dataset,
            data_collator=data_collator,
        )

        trainer.train()

    def generate(self, text, max_length=64):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        outputs = self.model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# ====================
# Usage
# ====================
generator = T5SentenceGeneratorScratch()
generator.train_from_excel("/content/Text and Keywords.xlsx", epochs=5, batch_size=2, learning_rate=5e-4)

# Example generation
#print(generator.generate("Your input text here"))


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss
50,7.010800
100,6.278900
150,6.241800
200,6.110500
250,5.926700
300,5.792100
350,5.831700
400,5.571100
450,5.439500
500,5.526500


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# Save everything after training
save_path = "/content/t5_scratch_model"

generator.model.save_pretrained(save_path)
generator.tokenizer.save_pretrained(save_path)

print(f"Model saved at {save_path}")


In [ ]:
import shutil

shutil.make_archive("/content/t5_scratch_model", 'zip', save_path)
from google.colab import files
files.download("/content/t5_scratch_model.zip")


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq

# ====================
# Custom Dataset
# ====================
class TextKeywordDataset(Dataset):
    def __init__(self, dataframe, tokenizer, source_max_len=128, target_max_len=64):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.source_max_len = source_max_len
        self.target_max_len = target_max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        source_text = str(self.data.iloc[idx, 0])  # assuming first col = text
        target_text = str(self.data.iloc[idx, 1])  # assuming second col = keywords

        source = self.tokenizer(
            source_text,
            padding="max_length",
            truncation=True,
            max_length=self.source_max_len,
            return_tensors="pt"
        )
        target = self.tokenizer(
            target_text,
            padding="max_length",
            truncation=True,
            max_length=self.target_max_len,
            return_tensors="pt"
        )

        labels = target["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100  # ignore padding in loss

        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "labels": labels,
        }

# ====================
# Main Trainer Class
# ====================
class T5SentenceGeneratorScratch:
    def __init__(self, model_name="t5-small"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)

        # Define config explicitly (important when training from scratch)
        from transformers import T5Config
        config = T5Config.from_pretrained(model_name)
        config.decoder_start_token_id = self.tokenizer.pad_token_id
        config.eos_token_id = self.tokenizer.eos_token_id
        config.pad_token_id = self.tokenizer.pad_token_id
        config.vocab_size = self.tokenizer.vocab_size

        # Initialize model from scratch
        self.model = T5ForConditionalGeneration(config)

    def train_from_excel(self, excel_path, epochs=5, batch_size=2, learning_rate=5e-4):
        # Load dataset
        df = pd.read_excel(excel_path)

        dataset = TextKeywordDataset(df, self.tokenizer)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        data_collator = DataCollatorForSeq2Seq(self.tokenizer, model=self.model)

        # Training arguments
        training_args = TrainingArguments(
            output_dir="./results",
            overwrite_output_dir=True,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            save_steps=1000,
            save_total_limit=2,
            learning_rate=learning_rate,
            logging_dir="./logs",
            logging_steps=50,
            report_to="none",  # disables wandb
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=dataset,
            data_collator=data_collator,
        )

        trainer.train()

    def generate(self, text, max_length=64):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        outputs = self.model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# ====================
# Usage
# ====================
generator = T5SentenceGeneratorScratch()
generator.train_from_excel("/content/Text and Keywords.xlsx", epochs=10, batch_size=32, learning_rate=5e-4)

# Example generation
#print(generator.generate("Your input text here"))

# Save everything after training
save_path = "/content/t5_scratch_model_Try2"

generator.model.save_pretrained(save_path)
generator.tokenizer.save_pretrained(save_path)

print(f"Model saved at {save_path}")


import shutil

shutil.make_archive("/content/t5_scratch_model_Try2", 'zip', save_path)
from google.colab import files
files.download("/content/t5_scratch_model_Try2.zip")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/pytho

Step,Training Loss
50,6.439000
100,5.510000
150,5.181400
200,4.931700
250,4.697100
300,4.651300
350,4.537500
400,4.472000
450,4.321500
500,4.259000


Model saved at /content/t5_scratch_model_Try2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>